## Imports

In [1]:
from pathlib import Path
import sys
import os

import numpy as np

import pycuda.autoinit
import pycuda.driver as cuda
from pycuda.compiler import SourceModule

In [2]:
from common import read_file_str, show_formatted_cpp, replace_constants_in_kernel

In [3]:
!cl

usage: cl [ option... ] filename... [ /link linkoption... ]


Microsoft (R) C/C++ Optimizing Compiler Version 19.43.34810 for x64
Copyright (C) Microsoft Corporation.  All rights reserved.



## Parameters

In [4]:
project_working_dir = str(Path(sys.path[0]).parent)
sys.path += [project_working_dir]
os.chdir(project_working_dir)

## Create dummy data for a mesh

In [5]:
vertices = np.array(
    [
        [0, 0, 0],
        [1, 0, 0],
        [2, 0, 0],
        [0, 0, 0],
        [1, 0.5, 0],
        [2, 0, 0],
    ],
    dtype=np.float32,
)

In [6]:
bend_relations = np.array([[0, 1, 2], [3, 4, 5]], dtype=np.uint32)

In [7]:
acceleration = vertices.copy() * 0

## Cuda Parameters

In [8]:
BLOCK_SIZE = 1024
NR_BLOCKS = (len(acceleration) + BLOCK_SIZE - 1) // BLOCK_SIZE

## Compile cuda kernel

In [9]:
cuda_code = read_file_str("./profiling/kernels/apply_bend.cu")

In [10]:
parameter_updates = {"BEND_THRESHOLD": 0.1, "BEND_WEIGHTING": 1}

In [11]:
cuda_code = replace_constants_in_kernel(cuda_code, parameter_updates)

In [12]:
show_formatted_cpp(cuda_code)

In [13]:
mod = SourceModule(cuda_code)

C:\Users\CYBORG\AppData\Local\Temp\ipykernel_16620\3464942708.py:1: UserWarning: The CUDA compiler succeeded, but said the following:
kernel.cu

  mod = SourceModule(cuda_code)


## Set-up memory for running kernel

In [14]:
apply_bend = mod.get_function("apply_bend")

In [15]:
nr_bend_relations = np.uint32(len(bend_relations))

### Allocate memory to gpu

In [16]:
assert acceleration.flatten().flags["C_CONTIGUOUS"]
assert vertices.flatten().flags["C_CONTIGUOUS"]
assert bend_relations.flatten().flags["C_CONTIGUOUS"]

In [17]:
acceleration_gpu = cuda.mem_alloc(acceleration.nbytes)
vertices_gpu = cuda.mem_alloc(vertices.nbytes)
bend_relations_gpu = cuda.mem_alloc(bend_relations.nbytes)

In [18]:
cuda.memcpy_htod(acceleration_gpu, acceleration.flatten())
cuda.memcpy_htod(vertices_gpu, vertices.flatten())
cuda.memcpy_htod(bend_relations_gpu, bend_relations.flatten())

## Create function

In [19]:
def apply_bend_kernel():
    apply_bend(
        acceleration_gpu,
        vertices_gpu,
        bend_relations_gpu,
        nr_bend_relations,
        block=(BLOCK_SIZE, 1, 1),
        grid=(NR_BLOCKS, 1, 1),
    )

## Check output is as expected

In [20]:
apply_bend_kernel()

In [21]:
cuda.memcpy_dtoh(vertices, vertices_gpu)
cuda.memcpy_dtoh(acceleration, acceleration_gpu)
cuda.memcpy_dtoh(bend_relations, bend_relations_gpu)

Expect first three to rest and second pair to accelerate down in y direction.

In [22]:
assert np.isclose(acceleration[:, 1], [0, 0, 0, 0, -0.5, 0]).all()

## Profile function

In [23]:
%timeit apply_bend_kernel()

13.9 µs ± 723 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
